In [8]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gc

In [9]:
class OneBitOptimizer(torch.optim.Optimizer):
    def __init__(self, params, lr=0.001, momentum=0.9):
        defaults = dict(lr=lr, momentum=momentum)
        super().__init__(params, defaults)
        self.basis_mats = {}
        for group in self.param_groups:
            for p in group['params']:
                if len(p.shape) < 2: # skip biases
                    continue
                row, col = p.shape
                n = max(2, int(2 ** np.ceil(np.log2(max(row, col))))) # need to find what dim we want for basis since not always power of 2 or square
                w_padded = torch.zeros((n, n), dtype=torch.float32, device=p.device)
                w_padded[:row, :col] = p.data
                w_flat = w_padded.view(1, -1)
                c_raw = self.hb_transform(w_flat).view(n, n)/(n * n)
                c = torch.sign(c_raw)
                c[c == 0] = 1
                self.state[p]["binary_coeffs"] = c
                self.state[p]["momentum_buffer"] = torch.zeros((n, n), dtype=torch.float32, device=p.device)
                w_dec_padded = self.hb_transform(c.view(1, -1)).view(n, n)
                w_dec = w_dec_padded[:row, :col]
                p.data.copy_(w_dec)
    
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                if len(p.shape) < 2: # skip biases
                    continue
                c = self.state[p]["binary_coeffs"]
                m = self.state[p]["momentum_buffer"]
                row, col = p.shape
                n = m.shape[0]
                grad_padded = torch.zeros((n, n), dtype=torch.float32, device=p.device)
                grad_padded[:row, :col] = p.grad.data
                m = group["momentum"] * m + (1 - group["momentum"]) * grad_padded
                m[row:, :] = 0
                m[:, col:] = 0
                self.state[p]["momentum_buffer"] = m
                mb = self.hb_transform(m.view(1, -1)).view(n, n)
                scores = mb * c
                # assuming we only update 1 bit per step
                min_wi = torch.argmin(scores)
                flipr = min_wi // n
                flipc = min_wi % n
                if scores[flipr, flipc] < 0:
                    c[flipr, flipc] *= -1
                self.state[p]["binary_coeffs"] = c
                w_dec_padded = self.hb_transform(c.view(1, -1)).view(n, n)/(n * n)
                w_dec = w_dec_padded[:row, :col]
                p.data.copy_(w_dec)
        return loss
    
    def hb_transform(self, x):
        if len(x.shape) == 1:
            x = x.view(1,-1)
        (m,n) = x.shape
        k = 1
        while 4**k < n:
            k += 1
        assert(4**k == n)
        x = x.reshape((m,) + (4,)*k)
        for i in range(k):
            x = x.sum(1+i,keepdim=True) - 2*x
        x = x.reshape((m,) + (2,2)*k)
        x = x.permute((0,) + tuple(2*i+1 for i in range(k)) + tuple(2*i+2 for i in range(k)))
        return x.reshape(m, 2**k, 2**k)

In [10]:
housing = fetch_california_housing()
X = housing.data
median_price = np.median(housing.target)
y = (housing.target > median_price).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset  = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=30, shuffle=False)

In [11]:
classes, counts = np.unique(y, return_counts=True)
total_samples = len(y)
print("Class distribution")
for cls, count in zip(classes, counts):
    percentage = (count / total_samples) * 100
    print(f"Class {cls}: {percentage:.2f}%")

Class distribution
Class 0: 50.01%
Class 1: 49.99%


In [12]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = OneBitOptimizer(model.parameters(), lr=0.01)
gc.collect()

2413

In [13]:
epochs = 20
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [14]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/20]
Training loss 14204331864222.76
Training accuracy 50.060562015503876 %
Testing loss 0.6933455235043238
Testing accuracy 49.685077519379846 %
Epoch [2/20]
Training loss 0.6932358975673831
Training accuracy 50.060562015503876 %
Testing loss 0.6933453529206819
Testing accuracy 49.685077519379846 %
Epoch [3/20]
Training loss 0.6932362036187519
Training accuracy 50.060562015503876 %
Testing loss 0.693345287771419
Testing accuracy 49.685077519379846 %
Epoch [4/20]
Training loss 0.6932358849764795
Training accuracy 50.060562015503876 %
Testing loss 0.6933451788716538
Testing accuracy 49.685077519379846 %
Epoch [5/20]
Training loss 0.6932356871606767
Training accuracy 50.060562015503876 %
Testing loss 0.6933462369407333
Testing accuracy 49.685077519379846 %
Epoch [6/20]
Training loss 0.6932361266871755
Training accuracy 50.060562015503876 %
Testing loss 0.6933452399490878
Testing accuracy 49.685077519379846 %
Epoch [7/20]
Training loss 0.6932352228566657
Training accuracy 50.06056